In [1]:
# Import Libraries
import kagglehub
import os
import glob
import pandas as pd
import numpy as np
import re
import base64
from PIL import Image
import cv2
import torch
import matplotlib.pyplot as plt

# Install YOLOv8
!pip install ultralytics -q
from ultralytics import YOLO

# Install PaddledOCR
!pip install paddleocr
!pip install paddlepaddle -f https://www.paddlepaddle.org.cn/whl/paddle/avx/stable.html
from paddleocr import PaddleOCR, draw_ocr

# Set OCR
ocr = PaddleOCR(use_angle_cls=True, lang='en')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.8/294.8 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 76.1 MB/s eta 0:00:00
  Created wheel for fire: filename=fire-0.7.0-py3-none-any.whl size=114249 sha256=94e0c8d51e94f6ae70c84ee46658ed6d9826c9747feccad6c5d8a57f0e6eca4e
  Stored in directory: /root

/usr/local/lib/python3.10/dist-packages/paddle/utils/cpp_extension/extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


download https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_det_infer.tar to /root/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer/en_PP-OCRv3_det_infer.tar


100%|██████████| 3910/3910 [00:15<00:00, 248.97it/s] 


download https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_infer.tar to /root/.paddleocr/whl/rec/en/en_PP-OCRv4_rec_infer/en_PP-OCRv4_rec_infer.tar


100%|██████████| 10000/10000 [00:19<00:00, 518.65it/s]


download https://paddleocr.bj.bcebos.com/dygraph_v2.0/ch/ch_ppocr_mobile_v2.0_cls_infer.tar to /root/.paddleocr/whl/cls/ch_ppocr_mobile_v2.0_cls_infer/ch_ppocr_mobile_v2.0_cls_infer.tar


100%|██████████| 2138/2138 [00:14<00:00, 151.89it/s]


[2025/05/03 12:10:14] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, use_gcu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='/root/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='/root/.paddleocr/whl/rec/en/en_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=6, max_text_l

In [2]:
# Download Dataset & Prepare Paths
dataset_path = kagglehub.dataset_download("bomaich/vnlicenseplate")
print("Path to dataset files:", dataset_path)

yaml_path = "/kaggle/working/my_yolo_dataset/dataset.yaml"
os.makedirs(os.path.dirname(yaml_path), exist_ok=True)

with open(yaml_path, "w") as f:
    f.write(f"""train: {dataset_path}/train/images
val: {dataset_path}/train/images  # use train set for val internally

nc: 1
names: ['license-plate']
""")
print("dataset.yaml created")

Path to dataset files: /kaggle/input/vnlicenseplate
dataset.yaml created


In [3]:
# Train YOLOv8 Model
model = YOLO("yolov8n.pt")
model.train(
    data=yaml_path,
    epochs=30,
    imgsz=640,
    batch=8
)

100%|██████████| 6.25M/6.25M [00:00<00:00, 86.2MB/s]


Ultralytics 8.3.124 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/kaggle/working/my_yolo_dataset/dataset.yaml, epochs=30, time=None, patience=100, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=

100%|██████████| 755k/755k [00:00<00:00, 17.6MB/s]


Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

100%|██████████| 5.35M/5.35M [00:00<00:00, 74.6MB/s]


...

      30/30      1.26G     0.4311     0.3377     0.8373          8        640: 100%|██████████| 46/46 [00:05<00:00,  8.96it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 23/23 [00:02<00:00,  7.88it/s]


                   all        365        622      0.996      0.973      0.985      0.909

30 epochs completed in 0.073 hours.
Optimizer stripped from runs/detect/train/weights/last.pt, 6.2MB
Optimizer stripped from runs/detect/train/weights/best.pt, 6.2MB

Validating runs/detect/train/weights/best.pt...
Ultralytics 8.3.124 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 23/23 [00:03<00:00,  7.15it/s]


                   all        365        622      0.996      0.973      0.985      0.909
Speed: 0.1ms preprocess, 1.7ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to runs/detect/train


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79224463ff40>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [4]:
# Load best weights
model = YOLO('/kaggle/working/runs/detect/train/weights/best.pt')

# Output directories
crop_output_dir = "/kaggle/working/plate_crops"
annotated_output_dir = "/kaggle/working/plate_annotated"
os.makedirs(crop_output_dir, exist_ok=True)
os.makedirs(annotated_output_dir, exist_ok=True)

In [5]:
# Run Inference + OCR
csv_rows = []
test_images = glob.glob(f"{dataset_path}/valid/images/*.jpg")

# OCR helper
def run_ocr(image):
    import numpy as np
    image = np.array(image)
    result = ocr.ocr(image, cls=True)

    if result is None or len(result) == 0 or result[0] is None:
        return ""

    texts = []
    for line in result:
        if line is None:
            continue
        for word_info in line:
            if word_info is None or word_info[1] is None:
                continue
            texts.append(word_info[1][0])

    return ' '.join(texts)


for image_path in test_images:
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    filename = os.path.basename(image_path)
    annotated_img = img_bgr.copy()

    results = model(img_rgb)

    for r in results:
        boxes = r.boxes.xyxy.cpu().numpy()
        confs = r.boxes.conf.cpu().numpy()

        for i, (box, conf) in enumerate(zip(boxes, confs)):
            x1, y1, x2, y2 = map(int, box)
            crop = img_rgb[y1:y2, x1:x2]
            crop_img = Image.fromarray(crop)

            # Save crop
            crop_filename = f"{filename.replace('.jpg','')}_plate{i+1}.jpg"
            crop_path = os.path.join(crop_output_dir, crop_filename)
            crop_img.save(crop_path)

            # ==== OCR with smart split for 2-line plates ====
            if crop_img.height > crop_img.width * 0.6:
                # Split top and bottom halves
                width, height = crop_img.size
                top_half = crop_img.crop((0, 0, width, height // 2))
                bottom_half = crop_img.crop((0, height // 2, width, height))

                text_top = run_ocr(top_half).strip()
                text_bot = run_ocr(bottom_half).strip()
                text = f"{text_top}-{text_bot}" if text_top and text_bot else text_top + text_bot
            else:
                text = run_ocr(crop_img).strip()

            # Annotate image
            cv2.rectangle(annotated_img, (x1, y1), (x2, y2), (0,255,0), 2)
            cv2.putText(annotated_img, text, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

            # Save annotated image
            annotated_path = os.path.join(annotated_output_dir, filename)
            cv2.imwrite(annotated_path, annotated_img)

            csv_rows.append({
                "filename": filename,
                "plate_number": text,
                "confidence": round(float(conf), 4),
                "annotated_path": annotated_path,
                "crop_path": crop_path
            })



0: 384x640 1 license-plate, 35.8ms
Speed: 2.4ms preprocess, 35.8ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)
[2025/05/03 12:15:06] ppocr DEBUG: dt_boxes num : 1, elapsed : 0.06907439231872559
[2025/05/03 12:15:06] ppocr DEBUG: cls num  : 1, elapsed : 0.034276723861694336
[2025/05/03 12:15:07] ppocr DEBUG: rec_res num  : 1, elapsed : 0.08860540390014648
[2025/05/03 12:15:07] ppocr DEBUG: dt_boxes num : 1, elapsed : 0.017904281616210938
[2025/05/03 12:15:07] ppocr DEBUG: cls num  : 1, elapsed : 0.007506132125854492
[2025/05/03 12:15:07] ppocr DEBUG: rec_res num  : 1, elapsed : 0.03981161117553711

0: 384x640 2 license-plates, 6.9ms
Speed: 2.7ms preprocess, 6.9ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
[2025/05/03 12:15:07] ppocr DEBUG: dt_boxes num : 1, elapsed : 0.06060671806335449
[2025/05/03 12:15:07] ppocr DEBUG: cls num  : 1, elapsed : 0.008108139038085938
[2025/05/03 12:15:07] ppocr DEBUG: rec_res num  : 1, elapsed : 0.03998351097106

In [6]:
# Save Raw Results
csv_df = pd.DataFrame(csv_rows)
csv_df.to_csv("/kaggle/working/plate_recognition_results.csv", index=False)
print("CSV saved: /kaggle/working/plate_recognition_results.csv")

CSV saved: /kaggle/working/plate_recognition_results.csv


In [7]:
# Evaluate YOLO Model (optional)

# Create a test-time dataset.yaml for evaluation
test_yaml_path = "/kaggle/working/test_dataset.yaml"
with open(test_yaml_path, "w") as f:
    f.write(f"""train: {dataset_path}/train/images
val: {dataset_path}/train/images
test: {dataset_path}/valid/images

nc: 1
names: ['license-plate']
""")

# Run evaluation on valid/ as a test set
metrics = model.val(data=test_yaml_path, split='test')
print("Test Set Evaluation Results:", metrics.results_dict)

Ultralytics 8.3.124 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 541.4±311.0 MB/s, size: 380.2 KB)


val: Scanning /kaggle/input/vnlicenseplate/valid/labels... 109 images, 0 backgrounds, 0 corrupt: 100%|██████████| 109/109 [00:00<00:00, 413.59it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/vnlicenseplate/valid is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:01<00:00,  3.68it/s]


                   all        109        125      0.992      0.992      0.995      0.915
Speed: 1.5ms preprocess, 3.2ms inference, 0.0ms loss, 3.4ms postprocess per image
Results saved to runs/detect/val
Test Set Evaluation Results: {'metrics/precision(B)': 0.991997143211866, 'metrics/recall(B)': 0.9916457582713969, 'metrics/mAP50(B)': 0.99476, 'metrics/mAP50-95(B)': 0.9149721908268997, 'fitness': 0.9229509717442098}


In [8]:
# Clean, Filter, Sort Plates
def clean_plate(text):
    return re.sub(r'[^a-zA-Z0-9]', '', str(text)).lower()

def is_valid_plate(text):
    return text.isalnum() and 4 <= len(text) <= 10

csv_df['clean_plate_number'] = csv_df['plate_number'].apply(clean_plate)
csv_df = csv_df[csv_df['clean_plate_number'].apply(is_valid_plate)]
csv_df = csv_df.sort_values(by="confidence", ascending=False)

In [9]:
# Generate HTML Report

html_blocks = []
html_blocks.append("<h2>Detected License Plates (Filtered & Sorted)</h2><div style='display:flex;flex-wrap:wrap;'>")

for _, row in csv_df.iterrows():
    with open(row['annotated_path'], "rb") as img_file:
        b64_img = base64.b64encode(img_file.read()).decode('utf-8')
    img_tag = f'<img src="data:image/jpeg;base64,{b64_img}" width="300" style="border:1px solid #ccc;">'

    html_blocks.append(f"""
        <div style="margin:10px;text-align:center;">
            {img_tag}<br>
            <strong>{row['clean_plate_number']}</strong><br>
            Confidence: {row['confidence']}
        </div>
    """)

html_blocks.append("</div>")
full_html = '\n'.join(html_blocks)

# Save HTML
html_output_path = "/kaggle/working/plate_detection_report.html"
with open(html_output_path, "w") as f:
    f.write(full_html)

print("HTML report with embedded images saved to:", html_output_path)

HTML report with embedded images saved to: /kaggle/working/plate_detection_report.html


In [10]:
# Load trained YOLOv8 model
yolo_model = YOLO("/kaggle/working/runs/detect/train/weights/best.pt")

# Save the full YOLO model
yolo_model.save("YOLOv8_plate_full.pt")